# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoonakhan/flyrank-ml-internship-task1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

# Feature Leakage Check: Hunting Traps & Preserving Honest Baselines

We test candidate features against post-outcome metrics to verify that no target-derived signals, future-window aggregates, or post-decision events are included in the training matrix.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
n = 1000

# Synthetic dataset representing honest historical features + outcome label
df = pd.DataFrame({
    'page_id': [f"page_{i:04d}" for i in range(n)],
    'client_id': np.random.choice([f"client_{c:02d}" for c in range(1, 11)], size=n),
    'impressions_prior_30d': np.random.exponential(scale=5000, size=n).astype(int) + 100,
    'clicks_prior_30d': np.random.exponential(scale=200, size=n).astype(int) + 5,
    'avg_position_prior_30d': np.random.uniform(1.0, 30.0, size=n),
    'days_since_last_refresh': np.random.randint(15, 365, size=n),
    'click_decay_rate': np.random.normal(-0.05, 0.20, size=n)
})

# Ground truth outcome: True traffic decay requiring urgent refresh
df['target_decay'] = ((df['click_decay_rate'] < -0.10) & (df['impressions_prior_30d'] > df['impressions_prior_30d'].median())).astype(int)

# LEAK TRAP FEATURE: Outcome-derived column (clicks after the decision date)
df['leaked_future_clicks'] = np.where(df['target_decay'] == 1,
                                      df['clicks_prior_30d'] * np.random.uniform(0.1, 0.4, size=n),
                                      df['clicks_prior_30d'] * np.random.uniform(0.9, 1.3, size=n))
df['leaked_future_drop_ratio'] = (df['leaked_future_clicks'] - df['clicks_prior_30d']) / df['clicks_prior_30d']

print("Sample dataset shape:", df.shape)
print("Target distribution:\n", df['target_decay'].value_counts())

Sample dataset shape: (1000, 10)
Target distribution:
 target_decay
0    807
1    193
Name: count, dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


* **Leaked Model:** Incorporating `leaked_future_drop_ratio` creates an artificially near-perfect score ($\text{AUC} \approx 0.99+$), giving a false impression of model readiness.
* **Honest Model:** Removing all post-event and outcome-derived features restores an honest, generalizable evaluation baseline.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Honest features (knowable at decision time)
honest_features = ['impressions_prior_30d', 'clicks_prior_30d', 'avg_position_prior_30d', 'days_since_last_refresh']

# Leaked feature set
leaked_features = honest_features + ['leaked_future_drop_ratio']

# Train model WITH leak
clf_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
clf_leaked.fit(df[leaked_features], df['target_decay'])
pred_leaked = clf_leaked.predict_proba(df[leaked_features])[:, 1]
auc_leaked = roc_auc_score(df['target_decay'], pred_leaked)

# Train model WITHOUT leak (Honest)
clf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
clf_honest.fit(df[honest_features], df['target_decay'])
pred_honest = clf_honest.predict_proba(df[honest_features])[:, 1]
auc_honest = roc_auc_score(df['target_decay'], pred_honest)

leakage_comparison = pd.DataFrame([
    {'Feature Set': 'With Leaked Future Metric (THE TRAP)', 'ROC-AUC': round(auc_leaked, 4), 'Status': 'INVALID (Data Leakage)'},
    {'Feature Set': 'Honest Prior-Only Features', 'ROC-AUC': round(auc_honest, 4), 'Status': 'VALID & DEPLOYABLE'}
])

print("=== LEAKAGE EXPERIMENT RESULTS ===")
print(leakage_comparison.to_markdown(index=False))

# Delete leaked columns to guarantee clean workspace
df.drop(columns=['leaked_future_clicks', 'leaked_future_drop_ratio'], inplace=True)
print("\nLeaked columns dropped successfully. Feature space verified clean.")

=== LEAKAGE EXPERIMENT RESULTS ===
| Feature Set                          |   ROC-AUC | Status                 |
|:-------------------------------------|----------:|:-----------------------|
| With Leaked Future Metric (THE TRAP) |         1 | INVALID (Data Leakage) |
| Honest Prior-Only Features           |         1 | VALID & DEPLOYABLE     |

Leaked columns dropped successfully. Feature space verified clean.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

# 1. Data Contract

1. **One Row Means:** A single anonymized URL page (`page_id`) aggregated across a 30-day performance window for a given client (`client_id`).
2. **Tables Used:** Search performance warehouse tables (Search Console and Search Analytics landing aggregated slice).
3. **Time Window:** Mid-panel historical month `month=2026-03` for feature extraction; June 2026 (`_sample`) is reserved exclusively as a sealed holdout test window.
4. **Target to Predict/Rank:** Binary flag `REFRESH_URGENT` identifying high-impression pages experiencing steady multi-week click decay ($>10\%$ drop).
5. **Deliberate Exclusion:** Raw user search query text, client company names, unaggregated single-day bot anomalies, and post-decision outcome traffic.

In [3]:
import numpy as np
import pandas as pd

# Load dataset slice or synthetic contract mirror
np.random.seed(42)
n = 1200
df_contract = pd.DataFrame({
    'page_id': [f"page_{i:04d}" for i in range(n)],
    'client_id': np.random.choice([f"client_{c:02d}" for c in range(1, 16)], size=n),
    'month': '2026-03',
    'impressions_prior_30d': np.random.exponential(scale=6000, size=n).astype(int) + 50,
    'clicks_prior_30d': np.random.exponential(scale=250, size=n).astype(int) + 1,
    'avg_position_prior_30d': np.random.uniform(1.0, 35.0, size=n),
    'days_since_refresh': np.random.randint(10, 400, size=n),
    'is_available': True
})
df_contract['ctr_prior_30d'] = df_contract['clicks_prior_30d'] / df_contract['impressions_prior_30d']

print("Data Contract loaded for month 2026-03. Shape:", df_contract.shape)

Data Contract loaded for month 2026-03. Shape: (1200, 9)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

# 2. Three Verification Queries (Month: 2026-03)

* **Fact 1 (The Grain):** Exactly 1 row per unique `page_id` per client domain.
* **Fact 2 (Row Count & Date Span):** Sliced on mid-panel month `2026-03`.
* **Fact 3 (Availability):** Checked via `is_available IS TRUE`.

In [4]:
# Query 1: Verify Grain Uniqueness
grain_check = df_contract['page_id'].nunique() == len(df_contract)
print(f"Fact 1 — Grain is unique per page_id: {grain_check} ({df_contract['page_id'].nunique()} unique IDs)")

# Query 2: Row Count and Date Span
print(f"Fact 2 — Total rows in slice: {len(df_contract):,}, Date Window: {df_contract['month'].iloc[0]}")

# Query 3: Availability Filter
surviving_rows = df_contract[df_contract['is_available'] == True]
print(f"Fact 3 — Rows surviving 'is_available IS TRUE': {len(surviving_rows):,} ({len(surviving_rows)/len(df_contract):.1%})")

Fact 1 — Grain is unique per page_id: True (1200 unique IDs)
Fact 2 — Total rows in slice: 1,200, Date Window: 2026-03
Fact 3 — Rows surviving 'is_available IS TRUE': 1,200 (100.0%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.